[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/S05_05_contrastes_causalidad.ipynb)

# S05_05 · Estadística con Python (2): contrastes de hipótesis, correlación y causalidad
**Sprint 5 · Matemáticas y estadística en código · ⏱ 60 min**

## Qué vas a aprender
1. A plantear y resolver un **contraste de hipótesis** (t de Student y chi-cuadrado) e interpretar el **p-valor**.
2. A medir el **tamaño del efecto** (no basta con "es significativo").
3. A calcular **correlaciones** y a ver por qué **correlación no es causalidad**.
4. A usar `statsmodels` para **controlar una variable de confusión** y a dar una idea de estadística bayesiana.

## Contexto TurisData
El cliente pregunta: *¿los festivos suben la ocupación?*, *¿el canal de reserva influye en las cancelaciones?*,
*¿hace bajar la ocupación el calor?* Contestaremos con números y con cautela. Datos sintéticos.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

SEMILLA = 42
diaria = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
diaria["mes"] = diaria["fecha"].dt.month
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
print(diaria.shape, reservas.shape)

## 1 · Contraste de hipótesis: la lógica
1. **Hipótesis nula (H0):** "no hay diferencia" (por ejemplo, la ocupación media es igual en festivos y en días normales).
2. Calculamos un **estadístico** y su **p-valor**: la probabilidad de ver una diferencia **al menos tan grande** *si H0 fuera cierta*.
3. Si `p < α` (habitualmente 0,05) **rechazamos H0**: los datos son difíciles de explicar por casualidad.

Ojo: el p-valor **no** es la probabilidad de que H0 sea cierta, ni mide la importancia práctica.

Comparamos la ocupación de días festivos y no festivos con el **t de Student de Welch** (no supone varianzas iguales).

In [ ]:
festivos = diaria.loc[diaria["festivo"] == 1, "ocupacion_pct"]
normales = diaria.loc[diaria["festivo"] == 0, "ocupacion_pct"]
print(f"Festivos: n={len(festivos)}, media={festivos.mean():.1f} · Normales: n={len(normales)}, media={normales.mean():.1f}")

### Ejercicio 1 · t de Student
Con `stats.ttest_ind(a, b, equal_var=False)` guarda en `p_festivo` el p-valor de comparar festivos y normales, en
`dif_medias` la diferencia de medias (festivos − normales, en puntos) y en `rechazar_h0` si se rechaza H0 con α = 0,05.

In [ ]:
# TODO: aplica ttest_ind a las dos series con equal_var=False; el p-valor es un atributo del resultado; compáralo con 0.05
...
assert p_festivo < 1e-20, f"El p-valor debería ser minúsculo (≈1e-30) y tienes {p_festivo:.3g}"
assert abs(dif_medias - 16.30) < 0.01, f"La diferencia de medias es 16.30 y tienes {dif_medias:.2f}"
assert rechazar_h0 is True, "Con ese p-valor se rechaza H0"
print(f"Correcto: diferencia = {dif_medias:.1f} puntos, p = {p_festivo:.2g} → rechazamos H0")

### Ejercicio 2 · Tamaño del efecto (d de Cohen)
Un p-valor pequeño dice que la diferencia **no parece casualidad**, no que sea **grande**. El **d de Cohen** mide la
diferencia en desviaciones típicas: `d = (media_a − media_b) / desviación_combinada`, con
`desviación_combinada = √(((n_a−1)·var_a + (n_b−1)·var_b) / (n_a + n_b − 2))`. (Regla orientativa: 0,2 pequeño, 0,5 medio, 0,8 grande.)
Guarda `d_cohen` (número).

In [ ]:
# TODO: calcula la desviación combinada con las varianzas (ddof=1) y los tamaños, y divide la diferencia de medias
...
assert abs(d_cohen - 1.84) < 0.02, f"d de Cohen ≈ 1.84 (un efecto muy grande) y tienes {d_cohen:.2f}"
print(f"Correcto: d = {d_cohen:.2f} → un festivo cambia la ocupación en casi 2 desviaciones típicas")

## 2 · Chi-cuadrado: dos variables categóricas
¿La **tasa de cancelación depende del canal** de reserva? Con una **tabla de contingencia** (recuento de cada
combinación) y `stats.chi2_contingency`. H0: canal y cancelación son independientes.

In [ ]:
tabla_canal = pd.crosstab(reservas["canal"], reservas["cancelada"])
print(tabla_canal)
print((100 * reservas.groupby("canal")["cancelada"].mean()).round(1).sort_values())

### Ejercicio 3 · Chi-cuadrado
Guarda en `p_canal` el p-valor de `chi2_contingency(tabla_canal)` (es el segundo valor que devuelve) y en
`canal_depende` si se rechaza la independencia con α = 0,05.

In [ ]:
# TODO: chi2, p, gl, esperados = stats.chi2_contingency(tabla_canal)
...
assert p_canal < 1e-30, f"El p-valor es minúsculo (≈8e-44) y tienes {p_canal:.3g}"
assert canal_depende is True, "Se rechaza la independencia: el canal se asocia con la cancelación"
print(f"Correcto: chi² = {chi2:.1f}, gl = {gl}, p = {p_canal:.2g}")

## 3 · Correlación
El **coeficiente de correlación de Pearson** `r` (entre −1 y 1) mide la relación **lineal** entre dos variables:
`r > 0` suben juntas, `r < 0` una sube cuando la otra baja, `r ≈ 0` no hay relación lineal. Spearman usa rangos
(robusto a extremos).

In [ ]:
print("Pearson:", round(diaria["ocupacion_pct"].corr(diaria["temperatura_c"]), 3))
print("Spearman:", round(stats.spearmanr(diaria["ocupacion_pct"], diaria["temperatura_c"]).statistic, 3))

diaria.plot.scatter(x="temperatura_c", y="ocupacion_pct", c="mes", colormap="twilight", s=8, figsize=(8, 4))
plt.xlabel("Temperatura (°C)")
plt.ylabel("Ocupación (%)")
plt.title("Ocupación frente a temperatura (el color es el mes)")
plt.tight_layout()
plt.show()

### Ejercicio 4 · Matriz de correlaciones
Calcula `matriz_corr`: la matriz de correlaciones (`DataFrame.corr()`) de las columnas `ocupacion_pct`,
`temperatura_c`, `festivo` y `precio_medio`. Guarda en `r_temp` la correlación de la ocupación con la temperatura
(redondeada a 2 decimales).

In [ ]:
# TODO: diaria[[...cuatro columnas...]].corr(); localiza la celda con .loc["ocupacion_pct", "temperatura_c"]
...
assert matriz_corr.shape == (4, 4), "La matriz debe ser 4 x 4"
assert r_temp == -0.63, f"La correlación ocupación-temperatura es -0.63 y tienes {r_temp}"
print(matriz_corr.round(2))

## 4 · Correlación no es causalidad
La correlación entre calor y ocupación es **−0,63**: cuanto más calor, menos ocupación. ¿Recomendarías al cliente
"enfriar" el hotel? No. Ocurre esto: en Canarias el invierno es temporada alta (viajeros que huyen del frío europeo)
y es la época **más fresca**. El **mes** actúa de **variable de confusión**: influye a la vez en la temperatura y en la
ocupación, creando una relación que no es un efecto directo del calor.

Para comprobarlo, comparamos dos regresiones con `statsmodels`: una **simple** y otra que **controla el mes**
(`C(mes)` crea una variable indicadora por mes) y el festivo.

In [ ]:
modelo_simple = smf.ols("ocupacion_pct ~ temperatura_c", data=diaria).fit()
print(modelo_simple.params.round(3).to_dict(), "p =", modelo_simple.pvalues["temperatura_c"])

### Ejercicio 5 · Controlar la confusión
Ajusta `modelo_control` con la fórmula `"ocupacion_pct ~ temperatura_c + festivo + C(mes)"`. Guarda en
`coef_temp_simple` el coeficiente de la temperatura del modelo simple, en `coef_temp_control` el del modelo con control
y en `p_temp_control` su p-valor. Después mira qué ha pasado con el efecto de la temperatura.

In [ ]:
# TODO: smf.ols(fórmula, data=diaria).fit(); los coeficientes están en .params y los p-valores en .pvalues
...
assert abs(coef_temp_simple + 1.86) < 0.01, f"El coeficiente simple es -1.86 y tienes {coef_temp_simple:.2f}"
assert abs(coef_temp_control) < 0.3, f"Al controlar el mes, el efecto de la temperatura casi desaparece (≈ -0.09) y tienes {coef_temp_control:.2f}"
assert p_temp_control > 0.05, f"Con control, el efecto de la temperatura ya no es significativo (p ≈ 0.48) y tienes p = {p_temp_control:.3f}"
print(f"Correcto: sin control, {coef_temp_simple:.2f} puntos por °C; controlando el mes, {coef_temp_control:.2f} (p = {p_temp_control:.2f}).")
print("El efecto del festivo se mantiene:", round(modelo_control.params["festivo"], 1), "puntos, p =", f"{modelo_control.pvalues['festivo']:.1g}")

**Conclusión para el cliente:** la temperatura *parece* importar solo porque coincide con la temporada. El efecto de los
festivos, en cambio, **resiste** el control (≈ +9 puntos). Con datos observacionales no podemos demostrar causalidad,
pero sí descartar explicaciones simples y decir cuáles son más creíbles.

## 5 · Un vistazo a la idea bayesiana
En vez de un p-valor, la estadística **bayesiana** parte de una creencia inicial y la actualiza con los datos. Para una
tasa (canceladas de un total) usamos una distribución **Beta**: con `c` canceladas de `n` y creencia inicial neutra,
la tasa "creíble" sigue `Beta(1 + c, 1 + n − c)`. Simulando de ella comparamos canales.

In [ ]:
def muestras_posterior(canal, n_muestras=100_000):
    rng = np.random.default_rng(SEMILLA)
    x = reservas.loc[reservas["canal"] == canal, "cancelada"]
    return rng.beta(1 + x.sum(), 1 + len(x) - x.sum(), n_muestras)

post_directo, post_web = muestras_posterior("directo"), muestras_posterior("web_intermediaria")
print("Tasa creíble directo (95 %):", np.percentile(post_directo, [2.5, 97.5]).round(3))
print("Tasa creíble web_intermediaria (95 %):", np.percentile(post_web, [2.5, 97.5]).round(3))

### Ejercicio 6 · ¿Qué probabilidad hay de que la web tenga más cancelaciones?
Guarda en `prob_web_mayor` la proporción de muestras en las que `post_web` es **mayor** que `post_directo`
(comparación elemento a elemento y `.mean()`).

In [ ]:
# TODO: (post_web > post_directo).mean()
...
assert prob_web_mayor > 0.999, f"Las distribuciones apenas se solapan: la probabilidad es ≈ 1.0 y tienes {prob_web_mayor}"
print(f"Correcto: P(web cancela más que directo) ≈ {prob_web_mayor:.3f}")

## 6 · Mini-reto integrador: ¿ocupación de invierno frente a verano?
Compara la ocupación de **invierno** (meses 12, 1, 2) con la de **verano** (6, 7, 8): guarda en `dif_estaciones`
la diferencia de medias (invierno − verano), en `ic_dif` el **intervalo de confianza del 95 %** de esa diferencia
(`resultado.confidence_interval()` del t-test de Welch, tupla `(low, high)`) y en `pregunta_causal` una cadena con tu
respuesta a: "¿podemos afirmar que el frío *provoca* más ocupación?" (`"si"` o `"no"`, y razona en una celda de texto).

In [ ]:
# TODO: filtra meses con .isin([...]); ttest_ind(invierno, verano, equal_var=False); resultado.confidence_interval()
...
assert abs(dif_estaciones - 16.38) < 0.02, f"La diferencia invierno-verano es 16.38 y tienes {dif_estaciones:.2f}"
assert abs(ic_dif[0] - 15.27) < 0.05 and abs(ic_dif[1] - 17.50) < 0.05, f"El IC debería ser ≈ (15.27, 17.50) y tienes {ic_dif}"
assert pregunta_causal == "no", "La respuesta prudente es 'no': la diferencia estacional no prueba que el frío sea la causa"
print(f"Correcto: invierno − verano = {dif_estaciones:.1f} puntos (IC 95 %: {ic_dif[0]:.1f} a {ic_dif[1]:.1f}).")

**Escribe aquí tu razonamiento (2–3 frases):** ¿por qué esta diferencia, aunque enorme y muy significativa, no prueba que
el frío cause más ocupación? ¿Qué otros factores cambian con la estación?

## 7 · Para reflexionar
1. ¿Qué diferencia hay entre "significativo" e "importante"? Usa el festivo y el d de Cohen como ejemplo.
2. Si el cliente te pide "demostrar" que subir el precio no reduce la ocupación, ¿qué te falta para hablar de causalidad? (pista: ¿qué haría un experimento?)
3. ¿Cuándo preferirías un intervalo bayesiano a un p-valor?

## 8 · Qué has aprendido
- Contrastes: `ttest_ind` (medias) y `chi2_contingency` (categorías); el p-valor y sus límites.
- El tamaño del efecto (d de Cohen) complementa al p-valor.
- Pearson/Spearman miden asociación, **no causa**; una variable de confusión (el mes) puede crear correlaciones engañosas.
- `statsmodels.ols` permite **controlar** variables y ver cómo cambian los coeficientes.
- Un primer contacto con la estadística bayesiana (Beta) para comparar tasas.